### Dataset 4: Transport & Logistics Data Rescue

In [1]:
import os
import re
import pandas as pd
import numpy as np


In [2]:
#  Load Raw Dataset
raw_csv_path = "../data/raw/track3_transport_logistics.csv"
if not os.path.exists(raw_csv_path):
    raw_csv_path = "track3_transport_logistics.csv"
df_raw = pd.read_csv(raw_csv_path)

In [3]:
print("Dataset Shape:", df_raw.shape)



Dataset Shape: (10400, 10)


In [4]:
print("First 10 Rows of Dataset: ")
df_raw.head(10)

First 10 Rows of Dataset: 


,trip_id,mandi_id,destination_warehouse,departure_time,arrival_time,transit_hours,distance,distance_unit,vehicle_no,driver_id
0,TRP006374,MANDI050,WH-Central,2026-04-30T20:08:22,01/05/2026 05:26,9.3,305.714532,miles,NaN,DRV264
1,TRP008869,MANDI029,WH-West,04-10-2026 04:15 AM,10/04/2026 09:51,NaN,263.5,km,NaN,DRV540
2,TRP000674,MANDI024,WH-South,18/07/2026 16:34,07-18-2026 10:46 PM,6.2,276.5,km,UP 50 BC 6882,DRV722
3,TRP000036,MANDI-042,WH-Central,08-13-2026 06:21 AM,14/08/2026,19.7,1070.7,km,RJ-52-CD-3274,DRV103
4,TRP008288,mandi_019,WH-South,07-30-2026 10:51 PM,07-31-2026 03:57 PM,17.1,728.9,km,DL-73-DF-1458,NaN
5,TRP003456,038,Export-Terminal,07-25-2026 08:53 PM,NaN,7.3,417.0,km,UP97-BC-6135,DRV255
6,TRP005236,mandi041,WH-West,2026-07-05 07:34:30,05/07/2026 22:28,14.9,671.9,km,PB-99-BC-2796,DRV297
7,TRP007075,023,WH-West,19/06/2026,2026-06-19T22:01:06,6.4,362.6,km,pb 48-DF-3590,NaN
8,TRP002222,mandi_033,WH-North,02-May-2026 10:12:36,02/05/2026 22:12,12.0,549.5,km,DL55-BC-7556,DRV662
9,TRP004565,MANDI054,WH-West,05/09/2026 10:36,05/09/2026 20:00,9.4,424.8,km,PB 83 DF 2210,DRV201


### Step 1: Raw CSV Data Profiling and Messiness Audit

**Problem Strategy**:
Before applying rescue transformations, we audit raw data messiness across:
1. `trip_id`: Identify duplicate trip records (400 duplicate rows).
2. `transit_hours`: Scan for negative sign logging anomalies (563 negative entries like `-4.3`, `-18.2`) and missing values.
3. `distance` & `distance_unit`: Detect miles vs km unit variations (1,553 miles) and missing units (1,032 missing).
4. `vehicle_no`: Inspect unstandardized Indian vehicle registration strings with inconsistent spacing, casing, and dashes.
5. `mandi_id`: Identify unformatted mandi primary keys (`mandi_019`, `038`, `mandi041`).


In [7]:
# Check for duplicates in the raw dataset
print("Raw Dataset Dimensions & Duplicate Scan:- ")
print()
print("Total Raw Rows:- ", len(df_raw))
print("Unique trip_id count:- ", df_raw['trip_id'].nunique())
print("Duplicate trip_id count:- ", df_raw.duplicated(subset=['trip_id']).sum())

Raw Dataset Dimensions & Duplicate Scan:- 

Total Raw Rows:-  10400
Unique trip_id count:-  10000
Duplicate trip_id count:-  400


In [10]:
# Check for negative and missing transit hours in the dataset
print("Transit Hours Anomalies & Missing Scan:- ")
print()
def parse_numeric(val):
    try:
        return float(val)
    except:
        return np.nan
raw_hours = df_raw['transit_hours'].apply(parse_numeric)
print("Negative transit hours count:- ", (raw_hours < 0).sum())
print("Missing transit hours count:- ", df_raw['transit_hours'].isnull().sum())

Transit Hours Anomalies & Missing Scan:- 

Negative transit hours count:-  563
Missing transit hours count:-  518


In [13]:
# Check for distance unit distribution in the dataset
print("Distance Unit Distribution Scan:- ")
print(df_raw['distance_unit'].value_counts(dropna=False))

Distance Unit Distribution Scan:- 
distance_unit
km       7815
miles    1553
NaN      1032
Name: count, dtype: int64


In [16]:
# Check for missing vehicle_no and driver_id in the dataset
print("Vehicle Number & Driver Missing Scan:- ")
print("Missing vehicle_no count:- ", df_raw['vehicle_no'].isnull().sum())
print("Missing driver_id count:- ", df_raw['driver_id'].isnull().sum())
print()
print("Sample raw vehicle_no strings:")
print()
print(df_raw['vehicle_no'].dropna().head(10).tolist())

Vehicle Number & Driver Missing Scan:- 
Missing vehicle_no count:-  1622
Missing driver_id count:-  1551

Sample raw vehicle_no strings:

['UP 50 BC 6882', 'RJ-52-CD-3274', 'DL-73-DF-1458', 'UP97-BC-6135', 'PB-99-BC-2796', 'pb 48-DF-3590', 'DL55-BC-7556', 'PB 83 DF 2210', 'PB 53 AB 4593', 'PB-65-AB-4841']


In [18]:
# Check for missing mandi_id in the dataset
print("Raw Mandi ID Sample:- ")
print(df_raw['mandi_id'].head(10).tolist())


Raw Mandi ID Sample:- 
['MANDI050', 'MANDI029', 'MANDI024', 'MANDI-042', 'mandi_019', '038', 'mandi041', '023', 'mandi_033', 'MANDI054']


### Step 2: Deduplication, Transit Hours Correction & Distance Standardisation (Miles -> KM)

**Problem Strategy**:
1. Deduplication: Drop exact duplicate rows based on `trip_id` (400 duplicate rows removed -> 10,000 unique trips preserved).
2. Transit Hours Correction: Fix negative sign logging anomalies using `abs()` (e.g., `-4.3` -> `4.3`) and create `is_negative_anomaly` flag.
3. Distance Standardisation: Convert all distances recorded in `miles` to kilometers (`km = miles * 1.60934`) and standardize `distance_unit` to `km`.


In [20]:
# Check Duplicates, Negative Transit Hours & Distance Units
print("Before Check:")
print("Total raw dataset rows:", len(df_raw))
print("Duplicate trip_id count:", df_raw.duplicated(subset=['trip_id']).sum())


Before Check:
Total raw dataset rows: 10400
Duplicate trip_id count: 400


In [21]:
# Parse Function to conver the value
def parse_numeric(val):
    try:
        return float(val)
    except:
        return np.nan

In [22]:
raw_hours = df_raw['transit_hours'].apply(parse_numeric)
print("Negative transit hours count:", (raw_hours < 0).sum())
print("Distance unit value counts:")
print(df_raw['distance_unit'].value_counts(dropna=False))

Negative transit hours count: 563
Distance unit value counts:
distance_unit
km       7815
miles    1553
NaN      1032
Name: count, dtype: int64


In [23]:
# Drop duplicate trip_id rows and create a new DataFrame
df = df_raw.drop_duplicates(subset=['trip_id']).copy()

In [24]:
# Clean transit_hours 
def clean_transit_hours(val):
    try:
        v = float(val)
        return abs(v)
    except:
        return np.nan

In [25]:
df['clean_transit_hours'] = df['transit_hours'].apply(clean_transit_hours)
df['is_negative_anomaly'] = df['transit_hours'].apply(
    lambda x: 1 if pd.notnull(x) and str(x).strip().startswith('-') else 0
)


In [27]:
# Parsing function to convert the distance value from  miles to km
def parse_distance(val):
    try:
        return float(val)
    except:
        return np.nan

In [28]:
df['raw_dist_num'] = df['distance'].apply(parse_distance)


In [29]:
# Convert miles to KM (1 mile = 1.60934 km)
df['clean_distance_km'] = np.where(
    df['distance_unit'].astype(str).str.lower() == 'miles',
    df['raw_dist_num'] * 1.60934,
    df['raw_dist_num']
)
df['clean_distance_unit'] = 'km'

##### BEFORE vs AFTER comparison

In [30]:
print("Before vs After Deduplication Row Counts:")
print("Raw rows:", len(df_raw), "| Deduplicated unique trips:", len(df))

Before vs After Deduplication Row Counts:
Raw rows: 10400 | Deduplicated unique trips: 10000


In [33]:
print("\nBefore vs After Transit Hours Anomaly Correction:")
print()
print("Negative transit hours count before:", (raw_hours < 0).sum(), "| After correction:", (df['clean_transit_hours'] < 0).sum())
print("Total negative sign anomalies flagged:", df['is_negative_anomaly'].sum())
print("Sample corrected transit hours:")
print(df[df['is_negative_anomaly'] == 1][['transit_hours', 'clean_transit_hours']].head(5))


Before vs After Transit Hours Anomaly Correction:

Negative transit hours count before: 563 | After correction: 0
Total negative sign anomalies flagged: 539
Sample corrected transit hours:
    transit_hours  clean_transit_hours
77           -4.3                  4.3
80           -3.7                  3.7
99          -18.2                 18.2
115         -20.9                 20.9
134         -23.4                 23.4


In [36]:
print("\nBefore vs After Distance Unit Conversion:")
print()
print("Miles records converted to KM:", (df['distance_unit'].astype(str).str.lower() == 'miles').sum())
print("Raw distance sum:", round(df['raw_dist_num'].sum(), 2), "| Rescued total KM distance sum:", round(df['clean_distance_km'].sum(), 2))
print("Sample distance conversion (Miles to KM):")
print(df[df['distance_unit'].astype(str).str.lower() == 'miles'][['distance', 'distance_unit', 'clean_distance_km']].head(5))


Before vs After Distance Unit Conversion:

Miles records converted to KM: 1497
Raw distance sum: 5499758.7 | Rescued total KM distance sum: 5870414.06
Sample distance conversion (Miles to KM):
              distance distance_unit  clean_distance_km
0           305.714532         miles         491.998625
19   608.7571687000001         miles         979.697262
20         630.0080569         miles        1013.897166
23  207.60005110000003         miles         334.099066
30         399.8522385         miles         643.498202


### Step 3: Vehicle Number Standardization, Mandi ID Normalization, and Timestamp Parsing

**Problem Strategy**:
1. Vehicle Number Standardization: Indian license plates contain mixed spaces, dashes, and casing (`UP 50 BC 6882`, `pb 48-DF-3590`, `DL55-BC-7556`). We use regex to standardize them into standard RTO format `SS-DD-XX-NNNN`.
2. Mandi ID Normalization: Standardize unformatted mandi IDs (`mandi_019`, `038`, `mandi041`) into `MANDIxxx` canonical keys.
3. Timestamp Parsing: Convert mixed departure and arrival timestamp strings into standardized `YYYY-MM-DD HH:MM:SS` strings.


### Befor Check of mandi_id

In [37]:
# Raw Vehicle Numbers, Mandi IDs & Timestamp Strings Scan
print("Raw Vehicle Numbers Sample:")
print(df['vehicle_no'].dropna().head(10).tolist())
print("\nRaw Mandi IDs Sample:")
print(df['mandi_id'].head(10).tolist())
print("\nRaw Departure Timestamps Sample:")
print(df['departure_time'].head(10).tolist())

Raw Vehicle Numbers Sample:
['UP 50 BC 6882', 'RJ-52-CD-3274', 'DL-73-DF-1458', 'UP97-BC-6135', 'PB-99-BC-2796', 'pb 48-DF-3590', 'DL55-BC-7556', 'PB 83 DF 2210', 'PB 53 AB 4593', 'PB-65-AB-4841']

Raw Mandi IDs Sample:
['MANDI050', 'MANDI029', 'MANDI024', 'MANDI-042', 'mandi_019', '038', 'mandi041', '023', 'mandi_033', 'MANDI054']

Raw Departure Timestamps Sample:
['2026-04-30T20:08:22', '04-10-2026 04:15 AM', '18/07/2026 16:34', '08-13-2026 06:21 AM', '07-30-2026 10:51 PM', '07-25-2026 08:53 PM', '2026-07-05 07:34:30', '19/06/2026', '02-May-2026 10:12:36', '05/09/2026 10:36']


In [38]:
# Vehicle Registration Number Standardization
def standardize_vehicle_no(val):
    if pd.isna(val) or val is None or str(val).strip() == '' or str(val).lower() == 'nan':
        return np.nan
    s = str(val).strip().upper()
    clean_str = re.sub(r'[^A-Z0-9]', '', s)
    m = re.match(r'^([A-Z]{2})(\d{2})([A-Z]{1,2})(\d{4})$', clean_str)
    if m:
        state, dist, series, num = m.groups()
        return f"{state}-{dist}-{series}-{num}"
    return s

In [39]:
df['clean_vehicle_no'] = df['vehicle_no'].apply(standardize_vehicle_no)


In [40]:
# Mandi ID Normalization
def standardize_mandi_id(val):
    if pd.isna(val) or val is None or str(val).strip() == '':
        return np.nan
    s = str(val).strip().upper()
    digits = ''.join(c for c in s if c.isdigit())
    return f"MANDI{int(digits):03d}" if digits else s


In [41]:
df['clean_mandi_id'] = df['mandi_id'].apply(standardize_mandi_id)


In [42]:
# Timestamp Parsing
def parse_timestamp(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    try:
        dt = pd.to_datetime(str(val).strip().replace('.', '-'), format='mixed', errors='coerce')
        if pd.notnull(dt):
            return dt.strftime('%Y-%m-%d %H:%M:%S')
    except:
        pass
    return np.nan

In [43]:
df['clean_departure_time'] = df['departure_time'].apply(parse_timestamp)
df['clean_arrival_time'] = df['arrival_time'].apply(parse_timestamp)

### Before vs After Checking

In [47]:
print("Before vs After Vehicle Number Standardization:")
df[df['vehicle_no'].notnull()][['vehicle_no', 'clean_vehicle_no']].head(10)

Before vs After Vehicle Number Standardization:


,vehicle_no,clean_vehicle_no
2,UP 50 BC 6882,UP-50-BC-6882
3,RJ-52-CD-3274,RJ-52-CD-3274
4,DL-73-DF-1458,DL-73-DF-1458
5,UP97-BC-6135,UP-97-BC-6135
6,PB-99-BC-2796,PB-99-BC-2796
7,pb 48-DF-3590,PB-48-DF-3590
8,DL55-BC-7556,DL-55-BC-7556
9,PB 83 DF 2210,PB-83-DF-2210
10,PB 53 AB 4593,PB-53-AB-4593
11,PB-65-AB-4841,PB-65-AB-4841


In [46]:
print("Before vs After Mandi ID Normalization:")
df[['mandi_id', 'clean_mandi_id']].head(10)

Before vs After Mandi ID Normalization:


,mandi_id,clean_mandi_id
0,MANDI050,MANDI050
1,MANDI029,MANDI029
2,MANDI024,MANDI024
3,MANDI-042,MANDI042
4,mandi_019,MANDI019
5,038,MANDI038
6,mandi041,MANDI041
7,023,MANDI023
8,mandi_033,MANDI033
9,MANDI054,MANDI054


In [49]:
print("\nBefore vs After Mandi ID Normalization:")
df[['mandi_id', 'clean_mandi_id']].head(10)


Before vs After Mandi ID Normalization:


,mandi_id,clean_mandi_id
0,MANDI050,MANDI050
1,MANDI029,MANDI029
2,MANDI024,MANDI024
3,MANDI-042,MANDI042
4,mandi_019,MANDI019
5,038,MANDI038
6,mandi041,MANDI041
7,023,MANDI023
8,mandi_033,MANDI033
9,MANDI054,MANDI054


In [50]:
print("Missing Timestamps Summary:")
print("Missing departure timestamps after parsing:", df['clean_departure_time'].isnull().sum())
print("Missing arrival timestamps after parsing:", df['clean_arrival_time'].isnull().sum())

Missing Timestamps Summary:
Missing departure timestamps after parsing: 0
Missing arrival timestamps after parsing: 1006


### Step 4: Timestamp & Transit Hours Imputation, Delay Analytics & Clean CSV Export

**Problem Strategy**:
1. Timestamp Reconstruction: Calculate missing `transit_hours` from `(arrival_time - departure_time)` where available. Reconstruct missing `arrival_time` using `departure_time + transit_hours` (**906 missing arrival timestamps recovered**).
2. Logistics Delay Analytics: Compute `expected_hours` (`distance_km / 40.0 km/h`), `delay_hours` (`clean_transit_hours - expected_hours`), and flag delayed trips (`is_delayed_flag = 1` if delay > 2 hours).
3. Final Export: Save cleaned logistics dataset to `data/processed/clean_transport_logistics.csv`.


##### Before check all missing values main part of a EDA

In [51]:
#  Missing Arrival Timestamps and Transit Hours Scan
print("Before Imputation Missing Counts:")
print("Missing departure timestamps count:", df['clean_departure_time'].isnull().sum())
print("Missing arrival timestamps count:", df['clean_arrival_time'].isnull().sum())
print("Missing transit_hours count:", df['clean_transit_hours'].isnull().sum())


Before Imputation Missing Counts:
Missing departure timestamps count: 0
Missing arrival timestamps count: 1006
Missing transit_hours count: 1006


In [52]:
# Convert departure and arrival string timestamps to datetime objects
s_dep = df['departure_time'].astype(str).str.strip().str.replace('.', '-', regex=False)
s_arr = df['arrival_time'].astype(str).str.strip().str.replace('.', '-', regex=False)
df['dep_dt'] = pd.to_datetime(s_dep, format='mixed', errors='coerce')
df['arr_dt'] = pd.to_datetime(s_arr, format='mixed', errors='coerce')

In [53]:
# Reconstruct missing transit_hours from (arr_dt - dep_dt)
mask_calc_hrs = df['clean_transit_hours'].isnull() & df['arr_dt'].notnull() & df['dep_dt'].notnull()
df.loc[mask_calc_hrs, 'clean_transit_hours'] = (df.loc[mask_calc_hrs, 'arr_dt'] - df.loc[mask_calc_hrs, 'dep_dt']).dt.total_seconds() / 3600.0
df['clean_transit_hours'] = df['clean_transit_hours'].abs()


In [54]:
# Reconstruct missing arrival timestamps from (dep_dt + clean_transit_hours)
mask_recon_arr = df['arr_dt'].isnull() & df['dep_dt'].notnull() & df['clean_transit_hours'].notnull()
df.loc[mask_recon_arr, 'arr_dt'] = df.loc[mask_recon_arr, 'dep_dt'] + pd.to_timedelta(df.loc[mask_recon_arr, 'clean_transit_hours'], unit='h')
df['clean_departure_time'] = df['dep_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['clean_arrival_time'] = df['arr_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')


In [55]:
# Logistics Delay Analytics
df['expected_hours'] = (df['clean_distance_km'] / 40.0).round(2)
df['delay_hours'] = (df['clean_transit_hours'] - df['expected_hours']).round(2)
df['is_delayed_flag'] = np.where(df['delay_hours'] > 2.0, 1, 0)

##### Before vs After Imputation

In [56]:
print("Before vs After Imputation Comparison:")
print("Missing arrival timestamps before:", df_raw['arrival_time'].isnull().sum(), "| After imputation:", df['clean_arrival_time'].isnull().sum())
print("Missing transit_hours before:", df_raw['transit_hours'].isnull().sum(), "| After imputation:", df['clean_transit_hours'].isnull().sum())

Before vs After Imputation Comparison:
Missing arrival timestamps before: 1053 | After imputation: 100
Missing transit_hours before: 518 | After imputation: 100


In [57]:
print("Logistics Delay Analytics Summary:")
print("Average actual transit hours:", round(df['clean_transit_hours'].mean(), 2))
print("Average expected transit hours:", round(df['expected_hours'].mean(), 2))
print("Total delayed trips (> 2 hrs buffer):", df['is_delayed_flag'].sum())

Logistics Delay Analytics Summary:
Average actual transit hours: 57.17
Average expected transit hours: 16.29
Total delayed trips (> 2 hrs buffer): 192


In [58]:
# Final Clean Schema Export
clean_cols = ['trip_id', 'clean_mandi_id', 'destination_warehouse', 'clean_departure_time', 
              'clean_arrival_time', 'clean_transit_hours', 'clean_distance_km', 
              'clean_vehicle_no', 'driver_id', 'is_negative_anomaly', 'delay_hours', 'is_delayed_flag']
df_clean = df[clean_cols].copy()
df_clean.columns = ['trip_id', 'mandi_id', 'destination_warehouse', 'departure_time', 
                    'arrival_time', 'transit_hours', 'distance_km', 
                    'vehicle_no', 'driver_id', 'is_negative_anomaly', 'delay_hours', 'is_delayed_flag']

In [59]:
# Save clean CSV file
os.makedirs("../data/processed", exist_ok=True)
export_path = "../data/processed/clean_transport_logistics.csv"
df_clean.to_csv(export_path, index=False)

In [60]:
print("Final Clean Dataset Shape:", df_clean.shape)

Final Clean Dataset Shape: (10000, 12)


In [61]:
print("\nFirst 10 Rows of Clean Transport Dataset:")
df_clean.head(10)


First 10 Rows of Clean Transport Dataset:


,trip_id,mandi_id,destination_warehouse,departure_time,arrival_time,transit_hours,distance_km,vehicle_no,driver_id,is_negative_anomaly,delay_hours,is_delayed_flag
0,TRP006374,MANDI050,WH-Central,2026-04-30 20:08:22,2026-01-05 05:26:00,9.3,491.998625,NaN,DRV264,0,-3.00,0
1,TRP008869,MANDI029,WH-West,2026-04-10 04:15:00,2026-10-04 09:51:00,4253.6,263.500000,NaN,DRV540,0,4247.01,1
2,TRP000674,MANDI024,WH-South,2026-07-18 16:34:00,2026-07-18 22:46:00,6.2,276.500000,UP-50-BC-6882,DRV722,0,-0.71,0
3,TRP000036,MANDI042,WH-Central,2026-08-13 06:21:00,2026-08-14 00:00:00,19.7,1070.700000,RJ-52-CD-3274,DRV103,0,-7.07,0
4,TRP008288,MANDI019,WH-South,2026-07-30 22:51:00,2026-07-31 15:57:00,17.1,728.900000,DL-73-DF-1458,NaN,0,-1.12,0
5,TRP003456,MANDI038,Export-Terminal,2026-07-25 20:53:00,2026-07-26 04:11:00,7.3,417.000000,UP-97-BC-6135,DRV255,0,-3.12,0
6,TRP005236,MANDI041,WH-West,2026-07-05 07:34:30,2026-05-07 22:28:00,14.9,671.900000,PB-99-BC-2796,DRV297,0,-1.90,0
7,TRP007075,MANDI023,WH-West,2026-06-19 00:00:00,2026-06-19 22:01:06,6.4,362.600000,PB-48-DF-3590,NaN,0,-2.67,0
8,TRP002222,MANDI033,WH-North,2026-05-02 10:12:36,2026-02-05 22:12:00,12.0,549.500000,DL-55-BC-7556,DRV662,0,-1.74,0
9,TRP004565,MANDI054,WH-West,2026-05-09 10:36:00,2026-05-09 20:00:00,9.4,424.800000,PB-83-DF-2210,DRV201,0,-1.22,0
